# Eksperimen 21 — Uji Statistik Baseline vs Main-Ingredient-Aware Reranking

Notebook ini membandingkan dua konfigurasi pada **38 query validation yang sama**:

- Baseline: `BASELINE_07_03`
- Improved: `MAIN050_W07_03`

Tujuan notebook ini bukan melakukan tuning baru, tetapi menguji apakah perbedaan performa yang sudah diamati cukup kuat secara statistik.

Uji yang digunakan:

1. **Exact McNemar's test** untuk `Hit@5` dan `Hit@10`, karena kedua metrik bersifat biner dan berpasangan per query.
2. **Paired bootstrap 95% confidence interval** untuk selisih `Recall@5` dan `MRR`.
3. **Paired sign-flip randomization test** untuk `Recall@5` dan `MRR`.

`Hit@5` diperlakukan sebagai metrik inferensial utama. Metrik lain diperlakukan sebagai analisis sekunder.

Interpretasi:
- `p < 0.05`: terdapat bukti statistik terhadap perbedaan pada sampel ini.
- `p >= 0.05`: belum ada bukti statistik yang cukup untuk menyatakan perbedaan.
- Hasil non-signifikan **tidak berarti kedua metode identik**; hanya berarti sampel yang tersedia belum memberi bukti yang cukup kuat.


In [1]:
import itertools
import numpy as np
import pandas as pd
from scipy.stats import binomtest

DETAIL_PATH = "hasil_main_ingredient_aware_validation_detail.csv"

BASELINE = "BASELINE_07_03"
IMPROVED = "MAIN050_W07_03"

SUMMARY_OUT = "hasil_uji_statistik_mainaware_validation.csv"
PAIRS_OUT = "hasil_uji_statistik_mainaware_pairs.csv"

BOOTSTRAP_ITER = 50000
SEED = 42


# ============================================================
# 1. Load paired validation results
# ============================================================
df = pd.read_csv(DETAIL_PATH)

required = {
    "config",
    "query_text",
    "hit_at_5",
    "hit_at_10",
    "recall_at_5",
    "reciprocal_rank",
}

missing = required - set(df.columns)
if missing:
    raise ValueError(f"Kolom hilang: {sorted(missing)}")

base = df[df["config"] == BASELINE].copy().set_index("query_text")

imp = df[df["config"] == IMPROVED].copy().set_index("query_text")

if len(base) == 0 or len(imp) == 0:
    raise ValueError("Baseline atau improved config tidak ditemukan pada file detail.")

if set(base.index) != set(imp.index):
    raise ValueError(
        "Query baseline dan improved tidak identik; paired test tidak valid."
    )

metrics = [
    "hit_at_5",
    "hit_at_10",
    "recall_at_5",
    "reciprocal_rank",
]

pairs = (
    base[metrics]
    .join(
        imp[metrics],
        lsuffix="_baseline",
        rsuffix="_improved",
    )
    .sort_index()
)

print("VALIDASI PASANGAN")
print(f"Jumlah paired query : {len(pairs)}")
print(f"Baseline config     : {BASELINE}")
print(f"Improved config     : {IMPROVED}")


# ============================================================
# 2. Exact McNemar test
# ============================================================
def exact_mcnemar(y_base, y_imp):
    """
    Exact McNemar via exact binomial test on discordant pairs.

    b = baseline hit, improved miss
    c = baseline miss, improved hit
    """
    y_base = np.asarray(y_base, dtype=int)
    y_imp = np.asarray(y_imp, dtype=int)

    both_hit = int(np.sum((y_base == 1) & (y_imp == 1)))
    b = int(np.sum((y_base == 1) & (y_imp == 0)))
    c = int(np.sum((y_base == 0) & (y_imp == 1)))
    both_miss = int(np.sum((y_base == 0) & (y_imp == 0)))

    discordant = b + c

    if discordant == 0:
        p_value = 1.0
    else:
        p_value = float(
            binomtest(
                k=min(b, c),
                n=discordant,
                p=0.5,
                alternative="two-sided",
            ).pvalue
        )

    return {
        "both_hit": both_hit,
        "baseline_only": b,
        "improved_only": c,
        "both_miss": both_miss,
        "discordant_pairs": discordant,
        "p_value": p_value,
    }


# ============================================================
# 3. Paired bootstrap CI
# ============================================================
def paired_bootstrap_diff(
    baseline_values,
    improved_values,
    n_iter=BOOTSTRAP_ITER,
    seed=SEED,
):
    baseline_values = np.asarray(
        baseline_values,
        dtype=float,
    )
    improved_values = np.asarray(
        improved_values,
        dtype=float,
    )

    diff = improved_values - baseline_values
    n = len(diff)

    rng = np.random.default_rng(seed)

    sample_idx = rng.integers(
        0,
        n,
        size=(n_iter, n),
    )

    boot_means = diff[sample_idx].mean(axis=1)

    ci_low, ci_high = np.quantile(
        boot_means,
        [0.025, 0.975],
    )

    return {
        "baseline_mean": float(baseline_values.mean()),
        "improved_mean": float(improved_values.mean()),
        "mean_difference": float(diff.mean()),
        "ci95_low": float(ci_low),
        "ci95_high": float(ci_high),
    }


# ============================================================
# 4. Exact paired sign-flip randomization test
# ============================================================
def paired_signflip_test(
    baseline_values,
    improved_values,
    seed=SEED,
    monte_carlo_iter=200000,
):
    """
    Two-sided paired randomization test.

    Jika jumlah pasangan dengan difference non-zero <= 20,
    seluruh 2^m sign assignments dihitung secara exact.
    Jika lebih besar, fallback ke Monte Carlo.
    """
    baseline_values = np.asarray(
        baseline_values,
        dtype=float,
    )
    improved_values = np.asarray(
        improved_values,
        dtype=float,
    )

    diff = improved_values - baseline_values
    nonzero = diff[np.abs(diff) > 1e-15]

    observed = abs(float(diff.mean()))
    m = len(nonzero)

    if m == 0:
        return {
            "nonzero_pairs": 0,
            "method": "exact",
            "p_value": 1.0,
        }

    denominator = len(diff)

    if m <= 20:
        extreme = 0
        total = 2**m

        for signs in itertools.product(
            [-1.0, 1.0],
            repeat=m,
        ):
            stat = abs(
                float(
                    np.dot(
                        np.asarray(signs),
                        nonzero,
                    )
                    / denominator
                )
            )

            if stat >= observed - 1e-15:
                extreme += 1

        p_value = extreme / total
        method = "exact"

    else:
        rng = np.random.default_rng(seed)

        signs = rng.choice(
            [-1.0, 1.0],
            size=(monte_carlo_iter, m),
        )

        stats = np.abs((signs * nonzero).sum(axis=1) / denominator)

        p_value = (np.sum(stats >= observed - 1e-15) + 1) / (monte_carlo_iter + 1)

        method = "monte_carlo"

    return {
        "nonzero_pairs": int(m),
        "method": method,
        "p_value": float(p_value),
    }


# ============================================================
# 5. Run tests
# ============================================================
results = []

for metric in [
    "hit_at_5",
    "hit_at_10",
]:
    bvals = pairs[f"{metric}_baseline"].to_numpy()

    ivals = pairs[f"{metric}_improved"].to_numpy()

    test = exact_mcnemar(
        bvals,
        ivals,
    )

    results.append(
        {
            "metric": metric,
            "test": "Exact McNemar",
            "baseline_mean": float(bvals.mean()),
            "improved_mean": float(ivals.mean()),
            "difference": float(ivals.mean() - bvals.mean()),
            "ci95_low": np.nan,
            "ci95_high": np.nan,
            "p_value": test["p_value"],
            "baseline_only": test["baseline_only"],
            "improved_only": test["improved_only"],
            "discordant_pairs": test["discordant_pairs"],
            "nonzero_pairs": np.nan,
        }
    )


for metric in [
    "recall_at_5",
    "reciprocal_rank",
]:
    bvals = pairs[f"{metric}_baseline"].to_numpy()

    ivals = pairs[f"{metric}_improved"].to_numpy()

    boot = paired_bootstrap_diff(
        bvals,
        ivals,
    )

    perm = paired_signflip_test(
        bvals,
        ivals,
    )

    results.append(
        {
            "metric": ("mrr" if metric == "reciprocal_rank" else metric),
            "test": (f"Paired bootstrap CI + {perm['method']} sign-flip"),
            "baseline_mean": boot["baseline_mean"],
            "improved_mean": boot["improved_mean"],
            "difference": boot["mean_difference"],
            "ci95_low": boot["ci95_low"],
            "ci95_high": boot["ci95_high"],
            "p_value": perm["p_value"],
            "baseline_only": np.nan,
            "improved_only": np.nan,
            "discordant_pairs": np.nan,
            "nonzero_pairs": perm["nonzero_pairs"],
        }
    )


result_df = pd.DataFrame(results)

result_df["significant_at_0_05"] = result_df["p_value"] < 0.05

result_df.to_csv(
    SUMMARY_OUT,
    index=False,
    encoding="utf-8-sig",
)

pairs.to_csv(
    PAIRS_OUT,
    encoding="utf-8-sig",
)


# ============================================================
# 6. Print results
# ============================================================
print("\n" + "=" * 100)
print("UJI STATISTIK BASELINE VS MAIN-INGREDIENT-AWARE")
print("=" * 100)

for _, r in result_df.iterrows():
    print(f"\nMetric       : {r['metric']}")
    print(f"Test         : {r['test']}")
    print(f"Baseline     : {r['baseline_mean']:.6f}")
    print(f"Improved     : {r['improved_mean']:.6f}")
    print(f"Difference   : {r['difference']:+.6f}")

    if not pd.isna(r["ci95_low"]):
        print(f"95% CI diff  : [{r['ci95_low']:.6f}, {r['ci95_high']:.6f}]")

    print(f"p-value      : {r['p_value']:.6f}")
    print(f"Significant? : {'YES' if r['significant_at_0_05'] else 'NO'}")

    if not pd.isna(r["baseline_only"]):
        print(f"Baseline-only hit : {int(r['baseline_only'])}")
        print(f"Improved-only hit : {int(r['improved_only'])}")


# ============================================================
# 7. Query-level Hit@5 changes
# ============================================================
hit5_changes = pairs[
    [
        "hit_at_5_baseline",
        "hit_at_5_improved",
    ]
].copy()

hit5_changes["change"] = np.select(
    [
        (
            (hit5_changes["hit_at_5_baseline"] == 0)
            & (hit5_changes["hit_at_5_improved"] == 1)
        ),
        (
            (hit5_changes["hit_at_5_baseline"] == 1)
            & (hit5_changes["hit_at_5_improved"] == 0)
        ),
    ],
    [
        "improved_only",
        "baseline_only",
    ],
    default="same",
)

print("\n" + "=" * 100)
print("QUERY YANG BERUBAH PADA HIT@5")
print("=" * 100)

changed = hit5_changes[hit5_changes["change"] != "same"]

if len(changed):
    print(changed.to_string())
else:
    print("Tidak ada perubahan Hit@5 per query.")


# ============================================================
# 8. Paper-safe interpretation
# ============================================================
hit5_row = result_df[result_df["metric"] == "hit_at_5"].iloc[0]

print("\n" + "=" * 100)
print("INTERPRETASI UNTUK PAPER")
print("=" * 100)

if hit5_row["p_value"] < 0.05:
    print("Perbedaan Hit@5 menunjukkan bukti statistik pada alpha=0.05.")
else:
    print(
        "Hit@5 meningkat secara numerik, tetapi exact McNemar "
        "tidak menunjukkan bukti statistik yang cukup pada alpha=0.05."
    )
    print(
        "Karena itu, paper sebaiknya menyebut observed/numerical improvement, "
        "bukan statistically significant improvement."
    )

print(
    "MRR dan Recall@5 juga perlu dilaporkan bersama effect estimate, "
    "95% CI, dan p-value agar arah trade-off terlihat."
)

print("\nOUTPUT")
print(f"- {SUMMARY_OUT}")
print(f"- {PAIRS_OUT}")

VALIDASI PASANGAN
Jumlah paired query : 38
Baseline config     : BASELINE_07_03
Improved config     : MAIN050_W07_03

UJI STATISTIK BASELINE VS MAIN-INGREDIENT-AWARE

Metric       : hit_at_5
Test         : Exact McNemar
Baseline     : 0.315789
Improved     : 0.368421
Difference   : +0.052632
p-value      : 0.500000
Significant? : NO
Baseline-only hit : 0
Improved-only hit : 2

Metric       : hit_at_10
Test         : Exact McNemar
Baseline     : 0.394737
Improved     : 0.394737
Difference   : +0.000000
p-value      : 1.000000
Significant? : NO
Baseline-only hit : 1
Improved-only hit : 1

Metric       : recall_at_5
Test         : Paired bootstrap CI + exact sign-flip
Baseline     : 0.276316
Improved     : 0.328947
Difference   : +0.052632
95% CI diff  : [0.000000, 0.131579]
p-value      : 0.500000
Significant? : NO

Metric       : mrr
Test         : Paired bootstrap CI + exact sign-flip
Baseline     : 0.212742
Improved     : 0.207554
Difference   : -0.005188
95% CI diff  : [-0.058943, 0.